# 04. Эффективность каналов привлечения

**Цель.** Понять, какие каналы привлечения приводят самых ценных пользователей. В архиве нет AppsFlyer и Amplitude, поэтому используем то, что есть:
- **канал** = `secondary_category_filled` из таблицы привлечения;
- **поведение в приложении** (аналог Amplitude) = процессы пользователей: проверка личности, открытие карты, переводы, платежи, депозиты;
- **ценность** = транзакции и покупки у партнёров.

**Что важно знать об данных (проверено в аудите):**
- У многих клиентов в таблице привлечения **несколько разных каналов** (даты нет, определить «первый» канал нельзя).
- Есть полные дубли строк и пустые значения канала.
- Стоимости привлечения нет, поэтому оцениваем **качество** пользователей, а не окупаемость канала.
- Ценность это объём операций, а не доход компании.

**План.**
1. Чистим привлечение, считаем, у скольких клиентов несколько каналов.
2. Считаем метрики качества на клиента (активация, активность за 30 дней, покупки у партнёров, объём операций, шаги воронки).
3. Сравниваем каналы с доверительными интервалами.
4. Проверяем устойчивость к правилу атрибуции (один канал, любое касание, дробные веса).
5. Убираем эффект месяца регистрации (старые клиенты успели сделать больше операций).
6. Смотрим воронку по процессам приложения и объём против качества.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

ACQ_FILE = DATA_DIR / "SAPP привлечения.csv"
TX_FILE = DATA_DIR / "SAPP транзакции.csv"
PP_FILE = DATA_DIR / "SAPP Покупки у партнеров.csv"
USERS_FILE = DATA_DIR / "SAPP пользователи.csv"
PROC_FILE = DATA_DIR / "SAPP Процессы пользователей в приложении.csv"

SUCCESS_STATUSES = ("Успешные", "AUSR0401", "AUSR0402")
MIN_CLIENTS = 1000            # каналы меньшего размера не интерпретируем
PROCESS_START = "2026-01-01"  # период данных, как у транзакций

con = duckdb.connect()
con.execute("SET memory_limit = '2GB'")
con.execute("SET threads = 2")
print("OK:", all(p.exists() for p in [ACQ_FILE, TX_FILE, PP_FILE, USERS_FILE, PROC_FILE]))

## 1. Привлечение: очистка и многоканальные клиенты

Удаляем строки без `customer_id` и полные дубли, пустой канал помечаем как `unknown` (это не то же самое, что `organic`).

In [ ]:
con.execute(f"""
    CREATE OR REPLACE TABLE acq AS
    SELECT DISTINCT
        CAST(customer_id AS BIGINT) AS customer_id,
        COALESCE(secondary_category_filled, 'unknown') AS ch
    FROM read_csv_auto('{ACQ_FILE.as_posix()}')
    WHERE customer_id IS NOT NULL
""")

raw_rows = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{ACQ_FILE.as_posix()}')").fetchone()[0]
print(f"Строк в файле: {raw_rows:,}")
print(con.execute("SELECT COUNT(*) AS pairs, COUNT(DISTINCT customer_id) AS customers FROM acq").df())

per_cust = con.execute("""
    SELECT n_ch, COUNT(*) AS customers FROM (
        SELECT customer_id, COUNT(*) AS n_ch FROM acq GROUP BY customer_id)
    GROUP BY n_ch ORDER BY n_ch
""").df()
per_cust["share"] = (per_cust["customers"] / per_cust["customers"].sum()).round(3)
per_cust

## 2. Метрики качества на уровне клиента

Для каждого клиента считаем:
- `activated`: была хотя бы одна успешная транзакция;
- `active_last30`: была операция в последние 30 дней данных;
- `partner_buy`: была покупка у партнёров;
- `sum_amt`: суммарный объём успешных операций (выбросы обрезаны по 99,9-му процентилю);
- шаги воронки из процессов приложения (статус `COMPLETED`): проверка личности, открытие карты, переводы, платежи, депозит;
- `reg_month`: месяц регистрации (по самой ранней записи, в таблице пользователей id не уникален).

In [ ]:
status_list = ", ".join(f"'{s}'" for s in SUCCESS_STATUSES)

con.execute(f"""
    CREATE OR REPLACE TABLE tx_raw AS
    SELECT CAST(customer_id AS BIGINT) AS customer_id,
           CAST(transaction_date AS DATE) AS d,
           ABS(CAST(transaction_sum AS DOUBLE)) AS amt
    FROM read_csv_auto('{TX_FILE.as_posix()}')
    WHERE transaction_status IN ({status_list}) AND customer_id IS NOT NULL
""")
cap = con.execute("SELECT quantile_cont(amt, 0.999) FROM tx_raw").fetchone()[0]
last_day = con.execute("SELECT MAX(d) FROM tx_raw").fetchone()[0]
con.execute(f"""
    CREATE OR REPLACE TABLE txc AS
    SELECT customer_id, COUNT(*) AS n_tx, SUM(LEAST(amt, {cap})) AS sum_amt, MAX(d) AS last_d
    FROM tx_raw GROUP BY customer_id
""")

con.execute(f"""
    CREATE OR REPLACE TABLE ppc AS
    SELECT CAST(customer_id AS BIGINT) AS customer_id, COUNT(*) AS n_pp
    FROM read_csv_auto('{PP_FILE.as_posix()}') WHERE customer_id IS NOT NULL GROUP BY 1
""")

con.execute(f"""
    CREATE OR REPLACE TABLE usr AS
    SELECT CAST(customer_id AS BIGINT) AS customer_id,
           strftime(MIN(CAST(reg_date AS DATE)), '%Y-%m') AS reg_month
    FROM read_csv_auto('{USERS_FILE.as_posix()}') GROUP BY 1
""")

con.execute(f"""
    CREATE OR REPLACE TABLE proc AS
    SELECT CAST(customer_id AS BIGINT) AS customer_id,
        MAX((process_code = 'LivenessProcess' AND status = 'COMPLETED')::INT)      AS liveness_ok,
        MAX((process_code = 'OpenCardProcess' AND status = 'COMPLETED')::INT)      AS card_opened,
        MAX((process_code IN ('P2pMoneyTransferProcessV2', 'DomesticMoneyTransferProcess',
             'ChildDomesticMoneyTransferProcess') AND status = 'COMPLETED')::INT)  AS transfer_ok,
        MAX((process_code IN ('PaymentProcess', 'PartnerPaymentProcess_V2',
             'AviataPaymentProcess_V2', 'ArbuzPaymentProcess_V2')
             AND status = 'COMPLETED')::INT)                                       AS payment_ok,
        MAX((process_code = 'OpenDepositProcess' AND status = 'COMPLETED')::INT)   AS deposit_opened
    FROM read_csv_auto('{PROC_FILE.as_posix()}')
    WHERE customer_id IS NOT NULL AND started_at >= TIMESTAMP '{PROCESS_START}'
    GROUP BY 1
""")

cv = con.execute(f"""
    SELECT a.customer_id, a.n_ch,
        COALESCE(t.n_tx, 0)                                  AS n_tx,
        COALESCE(t.sum_amt, 0)                               AS sum_amt,
        (t.n_tx IS NOT NULL)::INT                            AS activated,
        COALESCE((t.last_d > DATE '{last_day}' - 30)::INT, 0) AS active_last30,
        (p.n_pp IS NOT NULL)::INT                            AS partner_buy,
        COALESCE(pr.liveness_ok, 0)    AS liveness_ok,
        COALESCE(pr.card_opened, 0)    AS card_opened,
        COALESCE(pr.transfer_ok, 0)    AS transfer_ok,
        COALESCE(pr.payment_ok, 0)     AS payment_ok,
        COALESCE(pr.deposit_opened, 0) AS deposit_opened,
        u.reg_month
    FROM (SELECT customer_id, COUNT(*) AS n_ch FROM acq GROUP BY customer_id) a
    LEFT JOIN txc  t  USING (customer_id)
    LEFT JOIN ppc  p  USING (customer_id)
    LEFT JOIN proc pr USING (customer_id)
    LEFT JOIN usr  u  USING (customer_id)
""").df()

print(f"Клиентов: {len(cv):,} | последний день данных: {last_day}")
print("Без месяца регистрации:", cv["reg_month"].isna().mean().round(4))
cv[["activated", "active_last30", "partner_buy", "liveness_ok", "card_opened",
    "transfer_ok", "payment_ok", "deposit_opened"]].mean().round(3)

## 3. Сравнение каналов (клиенты ровно с одним каналом)

Самый «чистый» вариант: клиенту можно однозначно приписать канал. Для долей считаем доверительный интервал Уилсона (95%).

In [ ]:
acq = con.execute("SELECT * FROM acq").df()
pairs = acq.merge(cv, on="customer_id")
pairs["w_any"] = 1.0
pairs["w_frac"] = 1.0 / pairs["n_ch"]

METRICS = ["activated", "active_last30", "partner_buy"]

def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    den = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return centre - half, centre + half

single = pairs[pairs["n_ch"] == 1]
rows = []
for ch, g in single.groupby("ch"):
    n = len(g)
    if n < MIN_CLIENTS:
        continue
    row = {"Канал": ch, "Клиентов": n}
    for m in METRICS:
        lo, hi = wilson(g[m].sum(), n)
        row[m] = g[m].mean()
        row[m + "_lo"], row[m + "_hi"] = lo, hi
    row["Объём на активированного, тг"] = g.loc[g.activated == 1, "sum_amt"].mean()
    row["Активных за 30 дн., чел."] = int(g["active_last30"].sum())
    rows.append(row)

res = pd.DataFrame(rows).set_index("Канал").sort_values("activated", ascending=False)
overall = single[METRICS].mean()
print("Средние по всем клиентам с одним каналом:")
print(overall.round(3).to_string())
res[["Клиентов", "activated", "active_last30", "partner_buy",
     "Объём на активированного, тг", "Активных за 30 дн., чел."]].round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
order = res.index
for ax, m, title in zip(axes, ["activated", "active_last30"],
                        ["Доля активированных (была операция)", "Доля активных за последние 30 дней"]):
    vals = res.loc[order, m].values
    err = [vals - res.loc[order, m + "_lo"].values, res.loc[order, m + "_hi"].values - vals]
    ax.barh(order, vals * 100, xerr=np.array(err) * 100, capsize=3)
    ax.axvline(overall[m] * 100, linestyle="--", color="gray", label="Среднее")
    ax.set_title(title)
    ax.set_xlabel("%")
    ax.invert_yaxis()
    ax.legend()
plt.tight_layout()
plt.show()

## 4. Устойчивость к правилу атрибуции

У 60% и более клиентов несколько каналов, и мы не знаем, какой из них «настоящий». Считаем метрику тремя способами:
- **один канал**: только клиенты ровно с одним каналом;
- **любое касание**: клиент учитывается в каждом своём канале;
- **дробная атрибуция**: клиент делится поровну между своими каналами (вес 1/k).

Если ранжирование каналов похоже во всех трёх вариантах, выводам можно доверять.

In [ ]:
def weighted_rate(df, wcol, metric):
    return (df[metric] * df[wcol]).groupby(df["ch"]).sum() / df[wcol].groupby(df["ch"]).sum()

big = res.index
comp = {}
for m in ["activated", "active_last30"]:
    comp[(m, "один канал")] = weighted_rate(single.assign(w=1.0), "w", m)
    comp[(m, "любое касание")] = weighted_rate(pairs, "w_any", m)
    comp[(m, "дробная")] = weighted_rate(pairs, "w_frac", m)
comp = pd.DataFrame(comp).loc[big]
comp.columns = [f"{a} | {b}" for a, b in comp.columns]
comp.round(3)

In [ ]:
act = comp[[c for c in comp.columns if c.startswith("activated")]]
print("Корреляция рангов каналов по активации (Спирмен):")
print(act.rank().corr(method="spearman").round(2))
print()
print("Ранг канала (1 = лучший) по активации:")
act.rank(ascending=False).astype(int)

## 5. Поправка на месяц регистрации

Клиент, пришедший в январе, успел сделать больше, чем пришедший в мае. Если у каналов разный состав по месяцам, сравнение нечестное. Приводим все каналы к одному составу месяцев (прямая стандартизация): считаем долю активированных внутри каждого месяца и усредняем с одинаковыми весами месяцев.

In [ ]:
s = single.dropna(subset=["reg_month"])
month_w = s["reg_month"].value_counts(normalize=True)

def standardized(g, metric, min_cell=30):
    by_m = g.groupby("reg_month")[metric].agg(["mean", "size"])
    by_m = by_m[by_m["size"] >= min_cell]
    w = month_w.reindex(by_m.index)
    return (by_m["mean"] * w).sum() / w.sum()

adj = pd.DataFrame({
    "Сырая активация": res["activated"],
    "С поправкой на месяц": [standardized(s[s["ch"] == ch], "activated") for ch in res.index],
}, index=res.index)
adj["Разница"] = adj["С поправкой на месяц"] - adj["Сырая активация"]

mix = (s[s["ch"].isin(res.index)].groupby(["ch", "reg_month"]).size()
         .unstack(fill_value=0))
mix = mix.div(mix.sum(axis=1), axis=0)
print("Состав клиентов канала по месяцам регистрации (доли):")
print(mix.round(2).to_string())
print()
adj.sort_values("С поправкой на месяц", ascending=False).round(3)

## 6. Воронка по процессам приложения

Какая доля клиентов канала прошла ключевые шаги (завершённые процессы): проверку личности, открытие карты, перевод, платёж, открытие депозита.

In [ ]:
FUNNEL = ["liveness_ok", "card_opened", "transfer_ok", "payment_ok", "deposit_opened"]
fun = single[single["ch"].isin(res.index)].groupby("ch")[FUNNEL].mean().loc[res.index]
fun.loc["ВСЕ (среднее)"] = single[FUNNEL].mean()
fun.columns = ["Проверка личности", "Открыта карта", "Перевод", "Платёж", "Депозит"]
fun.round(3)

## 7. Объём против качества

Один канал может приводить много клиентов низкого качества, другой мало, но хороших. Смотрим, сколько **активных** пользователей даёт каждый канал и какова их доля.

In [ ]:
vq = res[["Клиентов", "activated", "active_last30", "Активных за 30 дн., чел."]].copy()
vq["Доля от всех активных"] = vq["Активных за 30 дн., чел."] / vq["Активных за 30 дн., чел."].sum()
vq["Доля от всех клиентов"] = vq["Клиентов"] / vq["Клиентов"].sum()
vq["Качество к среднему"] = vq["active_last30"] / overall["active_last30"]
vq.sort_values("Активных за 30 дн., чел.", ascending=False).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(vq["Клиентов"], vq["active_last30"] * 100, s=60)
for ch, r in vq.iterrows():
    ax.annotate(ch, (r["Клиентов"], r["active_last30"] * 100), textcoords="offset points",
                xytext=(5, 4), fontsize=9)
ax.axhline(overall["active_last30"] * 100, linestyle="--", color="gray")
ax.set_xscale("log")
ax.set_xlabel("Клиентов с одним каналом (лог. шкала)")
ax.set_ylabel("Активны за последние 30 дней, %")
ax.set_title("Объём и качество каналов")
plt.tight_layout()
plt.show()

## 8. Выводы (заполни своими словами по результатам)

1. **Данные:** клиентов с несколькими каналами ...%, поэтому ... Пустой канал (`unknown`) у ...%.
2. **Лучшие каналы по качеству:** ... (активация ...%, активность за 30 дней ...%).
3. **Крупные каналы с качеством ниже среднего:** ... Они дают много клиентов, но ...
4. **Слабые каналы:** ... Однако (проверь) выборки малы / различается месяц регистрации.
5. **Устойчивость:** ранжирование ... при смене правила атрибуции (корреляция рангов ...), поправка на месяц регистрации ... (изменила / не изменила порядок).
6. **Воронка:** где у каналов «проседает» путь (проверка личности, открытие карты, первый перевод): ...
7. **Рекомендации:** ... (например, пересмотреть бюджет на каналы с низкой активацией, усилить онбординг на шаге ..., проверить A/B-тестом).
8. **Ограничения:** нет стоимости привлечения и даты касания; у части клиентов несколько каналов; ценность это объём операций, а не доход; `customer_id` в пользователях не уникален; канал может отражать тип клиента, а не эффект самого канала (причинность не доказана).